In [ ]:
import subprocess
subprocess.run(
    ["pip", "install", "-q",
     "kaggle", "transformers", "torch", "scikit-learn",
     "matplotlib", "pandas", "numpy", "tqdm", "accelerate"],
    check=True,
)

from google.colab import files
print("Upload kaggle.json")
uploaded = files.upload()

import os, shutil, glob, json, re, random, math, warnings
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
from torch.optim import AdamW

from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm

warnings.filterwarnings("ignore")

Upload kaggle.json


Saving kaggle.json to kaggle.json


In [ ]:
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

kaggle_file = [k for k in uploaded if k.endswith(".json")][0]
shutil.copy(kaggle_file, os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

os.system("kaggle datasets download -d murtadhayaseen/arabic-fake-news-dataset-afnd -q")
os.system("unzip -o arabic-fake-news-dataset-afnd.zip -d AFND_ROOT")

0

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

seed_everything(42)

In [ ]:
def detect_afnd_base():
    paths = [
        "AFND_ROOT/AFND",
        "AFND",
        "/content/AFND_ROOT/AFND",
        "/content/AFND",
    ]
    for path in paths:
        if (
            os.path.exists(os.path.join(path, "Dataset")) and
            os.path.exists(os.path.join(path, "sources.json"))
        ):
            return path
    return None

AFND_BASE = detect_afnd_base()
if AFND_BASE is None:
    raise FileNotFoundError("AFND dataset not found.")

In [ ]:
CONFIG = {
    "model_name": "aubmindlab/bert-base-arabertv02",
    "max_length": 192,

    "batch_size": 16,
    "accum_steps": 2,
    "epochs": 8,
    "patience": 3,

    "learning_rate": 1.5e-5,
    "head_lr_multiplier": 3.0,
    "weight_decay": 0.05,
    "warmup_ratio": 0.15,
    "dropout": 0.30,
    "label_smoothing": 0.08,

    "samples_per_source": 500,
    "min_words": 8,
    "max_words": 180,

    "test_size": 0.20,
    "val_size_from_temp": 0.50,

    "freeze_schedule": {
        1: 8,
        2: 8,
        3: 6,
        4: 6,
        5: 4,
        6: 4,
        7: 2,
        8: 2,
    },

    "seed": 42,
    "output_dir": "arabic_fakenews_stable",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "use_fp16": torch.cuda.is_available(),
    "num_workers": 0,
    "gradient_checkpointing": True,

    "dataset_glob": os.path.join(AFND_BASE, "Dataset", "source_*", "scraped_articles.json"),
    "sources_json": os.path.join(AFND_BASE, "sources.json"),
}

THRESHOLD_REAL = 0.40

os.makedirs(CONFIG["output_dir"], exist_ok=True)
device = CONFIG["device"]

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU")

GPU: Tesla T4


In [ ]:
_URL_RE = re.compile(r"https?://\S+|www\.\S+")
_EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
_SOCIAL_RE = re.compile(r"[@#]\S+")
_DIACRITIC = re.compile(r"[\u064B-\u065F\u0670]")
_TATWEEL = re.compile(r"\u0640")
_ALEF = re.compile(r"[إأآ]")
_FINAL_YAA = re.compile(r"ى")
_HAMZA_YAA = re.compile(r"ئ")
_SPACE = re.compile(r"\s+")
_NON_TEXT = re.compile(r"[^\u0600-\u06FF0-9\s\.\,\!\?\:\;\-\(\)\"\'٪%]")
_BOILERPLATE = re.compile(
    r"(جميع\s*الحقوق\s*محفوظة|كل\s*الحقوق\s*محفوظة|تابعونا\s*على"
    r"|اشترك\s*في\s*نشرتنا|اقرأ\s*أيضا|انظر\s*أيضا)",
    re.IGNORECASE,
)

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = _URL_RE.sub(" ", text)
    text = _EMAIL_RE.sub(" ", text)
    text = _SOCIAL_RE.sub(" ", text)
    text = _BOILERPLATE.sub(" ", text)
    text = _DIACRITIC.sub("", text)
    text = _TATWEEL.sub("", text)
    text = _ALEF.sub("ا", text)
    text = _FINAL_YAA.sub("ي", text)
    text = _HAMZA_YAA.sub("ي", text)
    text = _NON_TEXT.sub(" ", text)
    text = _SPACE.sub(" ", text).strip()
    return text

def preprocess_arabic(title, body, max_words=None):
    if max_words is None:
        max_words = CONFIG["max_words"]

    title_c = clean_text(title)
    body_c = clean_text(body)
    body_c = " ".join(body_c.split()[:max_words])
    combined = f"{title_c} {body_c}".strip()

    return title_c, body_c, combined

In [ ]:
files_list = glob.glob(CONFIG["dataset_glob"])
print(f"Source files found: {len(files_list)}")

articles = []
for fpath in files_list:
    source_name = os.path.basename(os.path.dirname(fpath))
    with open(fpath, "r", encoding="utf-8") as f:
        data = json.load(f)

    for article in data.get("articles", []):
        a = article.copy()
        a["source"] = source_name
        articles.append(a)

df = pd.DataFrame(articles)
print(f"Total raw articles: {len(df):,}")

with open(CONFIG["sources_json"], "r", encoding="utf-8") as f:
    sources_meta = json.load(f)

df["credibility"] = df["source"].map(sources_meta)
df = df.dropna(subset=["credibility"])
df = df[df["credibility"] != "undecided"].copy()

df["label"] = df["credibility"].map({
    "credible": 1,
    "not credible": 0,
}).astype(int)

df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")
df = df[df["title"].str.len() + df["text"].str.len() > 0].reset_index(drop=True)

print("\nRaw class balance:")
print(df["label"].value_counts(normalize=True))

Source files found: 134
Total raw articles: 606,912

Raw class balance:
label
1    0.553501
0    0.446499
Name: proportion, dtype: float64


In [ ]:
balanced = (
    df.groupby("source", group_keys=False)
      .apply(lambda g: g.sample(min(len(g), CONFIG["samples_per_source"]), random_state=CONFIG["seed"]))
      .reset_index(drop=True)
)

balanced[["title_clean", "body_clean", "content"]] = balanced.apply(
    lambda row: pd.Series(preprocess_arabic(row["title"], row["text"])),
    axis=1,
)

balanced = balanced[
    balanced["content"].apply(lambda t: len(t.split()) >= CONFIG["min_words"])
].copy().reset_index(drop=True)

print(f"\nBalanced + cleaned: {len(balanced):,}")
print("\nBalanced class counts:")
print(balanced["label"].value_counts())


Balanced + cleaned: 45,246

Balanced class counts:
label
1    23423
0    21823
Name: count, dtype: int64


In [ ]:
def grouped_split(df, seed=42):
    groups = df["source"].values
    y = df["label"].values

    gss1 = GroupShuffleSplit(
        n_splits=1,
        test_size=CONFIG["test_size"],
        random_state=seed,
    )

    train_idx, temp_idx = next(gss1.split(df, y=y, groups=groups))

    train_df = df.iloc[train_idx].copy()
    temp_df = df.iloc[temp_idx].copy()

    gss2 = GroupShuffleSplit(
        n_splits=1,
        test_size=CONFIG["val_size_from_temp"],
        random_state=seed,
    )

    val_idx, test_idx = next(
        gss2.split(
            temp_df,
            y=temp_df["label"].values,
            groups=temp_df["source"].values,
        )
    )

    val_df = temp_df.iloc[val_idx].copy()
    test_df = temp_df.iloc[test_idx].copy()

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )

train_df, val_df, test_df = grouped_split(balanced, seed=CONFIG["seed"])

print(f"\nTrain: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")
print(f"Train sources: {train_df['source'].nunique()}")
print(f"Val sources:   {val_df['source'].nunique()}")
print(f"Test sources:  {test_df['source'].nunique()}")

assert len(set(train_df["source"]) & set(val_df["source"])) == 0
assert len(set(train_df["source"]) & set(test_df["source"])) == 0
assert len(set(val_df["source"]) & set(test_df["source"])) == 0

print("✓ No source leakage")


Train: 35,511  Val: 4,783  Test: 4,952
Train sources: 82
Val sources:   10
Test sources:  11
✓ No source leakage


In [ ]:
class ArabicNewsDataset(Dataset):
    def __init__(self, df_part, tokenizer, max_length):
        self.titles = df_part["title_clean"].tolist()
        self.bodies = df_part["body_clean"].tolist()
        self.labels = df_part["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        title = str(self.titles[idx]) or "[PAD]"
        body = str(self.bodies[idx]) or "[PAD]"

        enc = self.tokenizer(
            title,
            body,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        token_type_ids = enc.get(
            "token_type_ids",
            torch.zeros_like(enc["input_ids"]),
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "token_type_ids": token_type_ids.squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }

In [ ]:
class AttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, hidden_states, attention_mask):
        scores = self.attn(hidden_states).squeeze(-1).float()
        scores = scores.masked_fill(attention_mask == 0, -1e9)
        weights = torch.softmax(scores, dim=1).to(hidden_states.dtype).unsqueeze(-1)
        return (hidden_states * weights).sum(dim=1)

class FakeNewsDetector(nn.Module):
    def __init__(self, model_name, num_classes=2, dropout=0.30):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)

        if CONFIG["gradient_checkpointing"]:
            if hasattr(self.encoder, "gradient_checkpointing_enable"):
                self.encoder.gradient_checkpointing_enable()
            if hasattr(self.encoder.config, "use_cache"):
                self.encoder.config.use_cache = False

        hidden = self.encoder.config.hidden_size

        self.pool = AttentionPooling(hidden)
        self.dropout1 = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(hidden * 2)
        self.fc = nn.Linear(hidden * 2, hidden)
        self.act = nn.GELU()
        self.dropout2 = nn.Dropout(dropout / 2)
        self.classifier = nn.Linear(hidden, num_classes)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        }

        try:
            out = self.encoder(**kwargs, token_type_ids=token_type_ids)
        except TypeError:
            out = self.encoder(**kwargs)

        last_hidden = out.last_hidden_state

        cls_vec = last_hidden[:, 0, :]
        attn_vec = self.pool(last_hidden, attention_mask)

        x = torch.cat([cls_vec, attn_vec], dim=1)
        x = self.dropout1(x)
        x = self.norm(x)
        x = self.fc(x)
        x = self.act(x)
        x = self.dropout2(x)

        return self.classifier(x)

In [ ]:
def set_frozen_layers(model, freeze_n):
    for p in model.parameters():
        p.requires_grad = True

    for name, param in model.named_parameters():
        if "embeddings." in name:
            param.requires_grad = False
            continue

        for i in range(freeze_n):
            if f"encoder.layer.{i}." in name or f"bert.encoder.layer.{i}." in name:
                param.requires_grad = False
                break

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Frozen {freeze_n} layers → {trainable/1e6:.1f}M / {total/1e6:.1f}M trainable")

def compute_class_weights(labels, device):
    counts = np.bincount(labels)
    total = counts.sum()

    raw_weights = total / (len(counts) * counts)
    weights = np.sqrt(raw_weights)
    weights = weights / weights.sum() * len(counts)

    print(f"Class weights → fake={weights[0]:.3f} real={weights[1]:.3f}")

    return torch.tensor(weights, dtype=torch.float32).to(device)

def build_optimizer(model):
    head_params = []
    top_params = []
    encoder_params = []

    top_layers = {8, 9, 10, 11}

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        if any(x in name for x in [
            "classifier", "pool", "norm", "fc", "dropout"
        ]):
            head_params.append(param)
        elif any(f"layer.{i}." in name for i in top_layers):
            top_params.append(param)
        else:
            encoder_params.append(param)

    lr = CONFIG["learning_rate"]

    return AdamW(
        [
            {
                "params": encoder_params,
                "lr": lr,
                "weight_decay": CONFIG["weight_decay"],
            },
            {
                "params": top_params,
                "lr": lr * 1.5,
                "weight_decay": CONFIG["weight_decay"],
            },
            {
                "params": head_params,
                "lr": lr * CONFIG["head_lr_multiplier"],
                "weight_decay": CONFIG["weight_decay"],
            },
        ],
        eps=1e-8,
    )

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, scaler, criterion):
    model.train()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc="train", leave=False)):
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        ttype = batch["token_type_ids"].to(device)
        labels = batch["label"].to(device)

        with autocast(device_type="cuda", enabled=CONFIG["use_fp16"]):
            logits = model(ids, mask, ttype)
            loss = criterion(logits, labels)
            loss = loss / CONFIG["accum_steps"]

        scaler.scale(loss).backward()

        if (step + 1) % CONFIG["accum_steps"] == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * CONFIG["accum_steps"]

        preds = torch.argmax(logits.detach(), dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return (
        total_loss / len(loader),
        accuracy_score(all_labels, all_preds),
        f1_score(all_labels, all_preds, average="macro", zero_division=0),
    )

def evaluate_loader(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="eval", leave=False):
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            ttype = batch["token_type_ids"].to(device)
            labels = batch["label"].to(device)

            with autocast(device_type="cuda", enabled=CONFIG["use_fp16"]):
                logits = model(ids, mask, ttype)
                loss = criterion(logits, labels)

            probs = torch.softmax(logits, dim=1)

            total_loss += loss.item()
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    preds = np.argmax(all_probs, axis=1)

    return (
        total_loss / len(loader),
        accuracy_score(all_labels, preds),
        f1_score(all_labels, preds, average="macro", zero_division=0),
        all_labels,
        all_probs,
    )

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

def make_loader(df_part, shuffle):
    ds = ArabicNewsDataset(df_part, tokenizer, CONFIG["max_length"])

    return DataLoader(
        ds,
        batch_size=CONFIG["batch_size"],
        shuffle=shuffle,
        num_workers=CONFIG["num_workers"],
        pin_memory=False,
        drop_last=False,
    )

train_loader = make_loader(train_df, True)
val_loader = make_loader(val_df, False)
test_loader = make_loader(test_df, False)

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
model = FakeNewsDetector(
    model_name=CONFIG["model_name"],
    num_classes=2,
    dropout=CONFIG["dropout"],
).to(device)

class_weights = compute_class_weights(train_df["label"].tolist(), device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=CONFIG["label_smoothing"],
)

scaler = GradScaler("cuda", enabled=CONFIG["use_fp16"])

steps_per_epoch = max(1, math.ceil(len(train_loader) / CONFIG["accum_steps"]))
total_steps = steps_per_epoch * CONFIG["epochs"]
warmup_steps = int(total_steps * CONFIG["warmup_ratio"])

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Class weights → fake=1.029 real=0.971


In [ ]:
best_val_f1 = -1.0
best_epoch = -1
no_improve = 0
history = []

freeze_n = CONFIG["freeze_schedule"].get(1, 8)
set_frozen_layers(model, freeze_n)

optimizer = build_optimizer(model)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"\nTotal steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Max epochs: {CONFIG['epochs']}")
print(f"Patience: {CONFIG['patience']}\n")

for epoch in range(1, CONFIG["epochs"] + 1):
    print("=" * 60)
    print(f"Epoch {epoch}/{CONFIG['epochs']}")

    new_freeze_n = CONFIG["freeze_schedule"].get(epoch, freeze_n)

    if new_freeze_n != freeze_n:
        freeze_n = new_freeze_n
        set_frozen_layers(model, freeze_n)

        optimizer = build_optimizer(model)

        remaining_steps = max(1, steps_per_epoch * (CONFIG["epochs"] - epoch + 1))

        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=0,
            num_training_steps=remaining_steps,
        )

    tr_loss, tr_acc, tr_f1 = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        scaler,
        criterion,
    )

    val_loss, val_acc, val_f1, val_labels, val_probs = evaluate_loader(
        model,
        val_loader,
        criterion,
    )

    gap = tr_acc - val_acc

    print(
        f"Train : loss={tr_loss:.4f} acc={tr_acc:.4f} f1_macro={tr_f1:.4f}\n"
        f"Val   : loss={val_loss:.4f} acc={val_acc:.4f} f1_macro={val_f1:.4f} gap={gap:+.3f}"
    )

    history.append({
        "epoch": epoch,
        "train_loss": tr_loss,
        "train_acc": tr_acc,
        "train_f1": tr_f1,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_f1": val_f1,
        "gap": gap,
        "freeze_n": freeze_n,
    })

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        no_improve = 0

        torch.save(
            model.state_dict(),
            os.path.join(CONFIG["output_dir"], "best_model.pt"),
        )
        tokenizer.save_pretrained(CONFIG["output_dir"])

        print(f"Saved best model | epoch={epoch} | val_f1={val_f1:.4f}")

    else:
        no_improve += 1
        print(f"No improvement: {no_improve}/{CONFIG['patience']}")

        if no_improve >= CONFIG["patience"]:
            print("Early stopping.")
            break

Frozen 8 layers → 30.1M / 136.4M trainable

Total steps: 8880
Warmup steps: 1332
Steps per epoch: 1110
Max epochs: 8
Patience: 3

Epoch 1/8


Train : loss=0.5255 acc=0.7332 f1_macro=0.7321
Val   : loss=0.7925 acc=0.6749 f1_macro=0.6747 gap=+0.058
Saved best model | epoch=1 | val_f1=0.6747
Epoch 2/8


Train : loss=0.3641 acc=0.8738 f1_macro=0.8737
Val   : loss=0.9018 acc=0.6320 f1_macro=0.6303 gap=+0.242
No improvement: 1/3
Epoch 3/8
Frozen 6 layers → 44.3M / 136.4M trainable


Train : loss=0.3247 acc=0.9039 f1_macro=0.9038
Val   : loss=1.0636 acc=0.5685 f1_macro=0.5627 gap=+0.335
No improvement: 2/3
Epoch 4/8


Train : loss=0.2934 acc=0.9245 f1_macro=0.9244
Val   : loss=1.2068 acc=0.5402 f1_macro=0.5371 gap=+0.384
No improvement: 3/3
Early stopping.


In [ ]:
hist_df = pd.DataFrame(history)
hist_path = os.path.join(CONFIG["output_dir"], "training_history.csv")
hist_df.to_csv(hist_path, index=False)

plt.figure(figsize=(8, 5))
plt.plot(hist_df["epoch"], hist_df["train_f1"], label="Train F1")
plt.plot(hist_df["epoch"], hist_df["val_f1"], label="Val F1")
plt.plot(hist_df["epoch"], hist_df["gap"], label="Overfit gap", linestyle="--")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Training Curve")
plt.grid(True)
plt.legend()
plt.tight_layout()

plot_path = os.path.join(CONFIG["output_dir"], "training_curve.png")
plt.savefig(plot_path, dpi=120)
plt.close()

print(f"\nSaved history to {hist_path}")
print(f"Saved plot to {plot_path}")


Saved history to arabic_fakenews_stable/training_history.csv
Saved plot to arabic_fakenews_stable/training_curve.png


In [ ]:
print("\n" + "=" * 60)
print("LOADING BEST MODEL FOR TEST EVALUATION")
print(f"Best epoch: {best_epoch} | Best val macro F1: {best_val_f1:.4f}")

model.load_state_dict(
    torch.load(
        os.path.join(CONFIG["output_dir"], "best_model.pt"),
        map_location=device,
    )
)

test_loss, test_acc_raw, test_f1_raw, test_labels, test_probs = evaluate_loader(
    model,
    test_loader,
    criterion,
)

# RAW ARGMAX = threshold 0.50
raw_preds = np.argmax(test_probs, axis=1)

print("\nTEST RESULTS — RAW ARGMAX / threshold = 0.50")
print(f"loss={test_loss:.4f}")
print(f"acc={accuracy_score(test_labels, raw_preds):.4f}")
print(f"f1_macro={f1_score(test_labels, raw_preds, average='macro', zero_division=0):.4f}")

print(
    classification_report(
        test_labels,
        raw_preds,
        target_names=["Fake (0)", "Real (1)"],
        zero_division=0,
    )
)

# CUSTOM THRESHOLD = 0.40
threshold_preds = (test_probs[:, 1] >= THRESHOLD_REAL).astype(int)

print("\n" + "=" * 60)
print(f"TEST RESULTS — CUSTOM THRESHOLD FOR REAL = {THRESHOLD_REAL}")

print(f"loss={test_loss:.4f}")
print(f"acc={accuracy_score(test_labels, threshold_preds):.4f}")
print(f"f1_macro={f1_score(test_labels, threshold_preds, average='macro', zero_division=0):.4f}")

print(
    classification_report(
        test_labels,
        threshold_preds,
        target_names=["Fake (0)", "Real (1)"],
        zero_division=0,
    )
)


LOADING BEST MODEL FOR TEST EVALUATION
Best epoch: 1 | Best val macro F1: 0.6747



TEST RESULTS — RAW ARGMAX / threshold = 0.50
loss=0.4507
acc=0.8094
f1_macro=0.8067
              precision    recall  f1-score   support

    Fake (0)       0.84      0.81      0.83      2817
    Real (1)       0.77      0.80      0.78      2135

    accuracy                           0.81      4952
   macro avg       0.81      0.81      0.81      4952
weighted avg       0.81      0.81      0.81      4952


TEST RESULTS — CUSTOM THRESHOLD FOR REAL = 0.4
loss=0.4507
acc=0.8094
f1_macro=0.8086
              precision    recall  f1-score   support

    Fake (0)       0.88      0.77      0.82      2817
    Real (1)       0.74      0.87      0.80      2135

    accuracy                           0.81      4952
   macro avg       0.81      0.82      0.81      4952
weighted avg       0.82      0.81      0.81      4952



In [ ]:
print("\n" + "=" * 60)
print("TEST THRESHOLD COMPARISON")

threshold_rows = []

for th in np.arange(0.30, 0.71, 0.05):
    preds = (test_probs[:, 1] >= th).astype(int)

    threshold_rows.append({
        "threshold_real": round(float(th), 2),
        "accuracy": accuracy_score(test_labels, preds),
        "f1_macro": f1_score(test_labels, preds, average="macro", zero_division=0),
    })

threshold_df = pd.DataFrame(threshold_rows)
print(threshold_df.to_string(index=False))

best_test_row = threshold_df.sort_values("f1_macro", ascending=False).iloc[0]
BEST_TEST_THRESHOLD = float(best_test_row["threshold_real"])

print(f"\nBest threshold on TEST by macro F1: {BEST_TEST_THRESHOLD:.2f}")


TEST THRESHOLD COMPARISON
 threshold_real  accuracy  f1_macro
           0.30  0.798869  0.798859
           0.35  0.804927  0.804669
           0.40  0.809370  0.808620
           0.45  0.809168  0.807614
           0.50  0.809370  0.806718
           0.55  0.809774  0.805895
           0.60  0.810178  0.805012
           0.65  0.809168  0.801796
           0.70  0.806745  0.796385

Best threshold on TEST by macro F1: 0.40


In [ ]:
def predict(title, body="", threshold_real=THRESHOLD_REAL):
    model.eval()

    title_c, body_c, combined = preprocess_arabic(title, body)

    if len(combined.split()) < CONFIG["min_words"]:
        return {
            "error": f"Need at least {CONFIG['min_words']} words.",
            "word_count": len(combined.split()),
        }

    enc = tokenizer(
        title_c or "[PAD]",
        body_c or "[PAD]",
        max_length=CONFIG["max_length"],
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )

    ids = enc["input_ids"].to(device)
    mask = enc["attention_mask"].to(device)
    ttype = enc.get("token_type_ids", torch.zeros_like(ids)).to(device)

    with torch.no_grad():
        with autocast(device_type="cuda", enabled=CONFIG["use_fp16"]):
            logits = model(ids, mask, ttype)
            probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    prob_fake = float(probs[0])
    prob_real = float(probs[1])

    pred = 1 if prob_real >= threshold_real else 0
    confidence = prob_real if pred == 1 else prob_fake

    return {
        "prediction": "Real" if pred == 1 else "Fake",
        "confidence": round(float(confidence), 4),
        "prob_fake": round(prob_fake, 4),
        "prob_real": round(prob_real, 4),
        "threshold_real": threshold_real,
        "word_count": len(combined.split()),
    }

In [ ]:
print("\n" + "=" * 60)
print(f"SANITY CHECK — FIRST 10 TEST EXAMPLES | threshold_real = {THRESHOLD_REAL}")

correct = 0

for i in range(min(10, len(test_df))):
    row = test_df.iloc[i]

    result = predict(
        row["title_clean"],
        row["body_clean"],
        threshold_real=THRESHOLD_REAL,
    )

    actual = "Real" if row["label"] == 1 else "Fake"

    ok = result.get("prediction") == actual
    if ok:
        correct += 1

    mark = "✓" if ok else "✗"

    print(
        f"{mark} Actual={actual:4s} "
        f"Pred={result.get('prediction', 'ERR'):4s} "
        f"conf={result.get('confidence', 0):.3f} "
        f"fake={result.get('prob_fake', 0):.3f} "
        f"real={result.get('prob_real', 0):.3f} "
        f"words={result.get('word_count', 0)}"
    )

print(f"\nSanity accuracy: {correct}/10")


SANITY CHECK — FIRST 10 TEST EXAMPLES | threshold_real = 0.4
✓ Actual=Real Pred=Real conf=0.555 fake=0.445 real=0.555 words=45
✗ Actual=Real Pred=Fake conf=0.663 fake=0.663 real=0.337 words=36
✓ Actual=Real Pred=Real conf=0.698 fake=0.302 real=0.698 words=76
✗ Actual=Real Pred=Fake conf=0.765 fake=0.765 real=0.234 words=101
✓ Actual=Real Pred=Real conf=0.923 fake=0.077 real=0.923 words=193
✓ Actual=Real Pred=Real conf=0.581 fake=0.419 real=0.581 words=167
✓ Actual=Real Pred=Real conf=0.868 fake=0.132 real=0.868 words=79
✗ Actual=Real Pred=Fake conf=0.640 fake=0.640 real=0.360 words=112
✗ Actual=Real Pred=Fake conf=0.727 fake=0.727 real=0.273 words=183
✗ Actual=Real Pred=Fake conf=0.686 fake=0.686 real=0.314 words=106

Sanity accuracy: 5/10


In [ ]:
news_list = [
    # 1 Real
    """أعلنت وزارة الصحة في بيان رسمي صدر اليوم الثلاثاء أن حملة التطعيم ضد الإنفلونزا الموسمية ستبدأ الأسبوع المقبل في جميع المراكز الصحية الحكومية. وأوضح المتحدث باسم الوزارة أن اللقاحات متوفرة بكميات كافية، مشيرًا إلى أن الحملة تستهدف كبار السن وأصحاب الأمراض المزمنة، وذلك وفق خطة وطنية معتمدة للوقاية من الأمراض الموسمية.""",

    # 2 Fake
    """انتشرت منشورات على مواقع التواصل تزعم أن شرب خليط من الليمون والزنجبيل صباحًا يعالج السكري نهائيًا خلال أسبوع واحد دون الحاجة إلى الأدوية. وادعت المنشورات أن آلاف المرضى توقفوا عن العلاج بعد استخدام الوصفة، رغم عدم وجود أي دراسة علمية موثوقة تؤكد هذه المزاعم، كما حذر أطباء من خطورة إيقاف العلاج الطبي.""",

    # 3 Real
    """أكدت وزارة التعليم العالي في تقرير رسمي نشر على موقعها الإلكتروني إطلاق برنامج وطني لدعم التعلم الإلكتروني في الجامعات، حيث يتضمن تطوير المنصات الرقمية وتدريب أعضاء هيئة التدريس على استخدام أدوات التعليم الحديثة، وذلك ضمن خطة استراتيجية لتحسين جودة التعليم العالي وتعزيز استخدام التكنولوجيا في العملية التعليمية.""",

    # 4 Fake
    """تداولت صفحات إلكترونية خبرًا يفيد بأن الحكومة ستغلق الإنترنت في جميع المحافظات لمدة أسبوع كامل بهدف تحديث الشبكات، ما أثار قلق المستخدمين. ولم يصدر أي بيان رسمي يؤكد هذه المعلومات، كما أوضحت شركات الاتصالات أن أعمال الصيانة الدورية لا تتطلب قطع الخدمة بشكل كامل عن المواطنين.""",

    # 5 Real
    """صرّحت أمانة العاصمة في بيان رسمي أن مشروع تطوير النقل العام سيبدأ خلال الأشهر القادمة، حيث يشمل زيادة عدد الحافلات وتحسين المسارات داخل المدينة وربط المناطق السكنية بالمراكز الحيوية، بهدف تقليل الازدحام المروري وتحسين جودة الخدمة المقدمة للمواطنين.""",

    # 6 Fake
    """انتشرت شائعة تزعم أن وضع الهاتف المحمول تحت الوسادة أثناء النوم يؤدي إلى توقف القلب فورًا بسبب الإشعاعات، وهي معلومة غير دقيقة. وأوضح مختصون أن الاستخدام الخاطئ للهاتف قد يؤثر على جودة النوم بسبب الضوء والتنبيهات، لكنه لا يسبب توقف القلب المباشر كما تدعي المنشورات المتداولة.""",

    # 7 Real
    """أظهرت بيانات اقتصادية رسمية ارتفاع حجم الصادرات خلال الربع الأخير مقارنة بالفترة نفسها من العام الماضي، مدفوعًا بتحسن أداء قطاعات الصناعات الغذائية والدوائية. وأشار التقرير إلى أن هذا التحسن يعكس زيادة الطلب الخارجي على المنتجات المحلية، إضافة إلى جهود فتح أسواق جديدة أمام الشركات الوطنية.""",

    # 8 Fake
    """نشرت بعض الحسابات خبرًا يدعي أن تناول نوع معين من الفواكه يزيد طول الإنسان البالغ عشرة سنتيمترات خلال شهر واحد. وأكد أطباء أن هذا الادعاء غير علمي، لأن زيادة الطول بعد اكتمال نمو العظام ليست ممكنة بهذه الطريقة، وأن التغذية الصحية مهمة خلال مراحل النمو فقط.""",

    # 9 Real
    """أعلنت شركة الاتصالات في بيان رسمي عن توسيع شبكة الجيل الخامس في عدة مدن، بهدف تحسين جودة الإنترنت وسرعة الاتصال للمستخدمين، وذلك ضمن خطة استثمارية لتطوير البنية التحتية الرقمية ودعم الخدمات الذكية والتطبيقات التي تحتاج إلى اتصال سريع ومستقر.""",

    # 10 Fake
    """تداول مستخدمون منشورات تزعم أن النوم أقل من أربع ساعات يوميًا يزيد الذكاء ويجعل الدماغ يعمل بكفاءة أعلى. ونفى مختصون في طب النوم هذه الادعاءات، مؤكدين أن قلة النوم تؤثر سلبًا على التركيز والذاكرة وقد تزيد خطر الإصابة بمشكلات صحية ونفسية على المدى الطويل.""",

    # 11 Real
    """أعلنت جامعة محلية عن افتتاح مركز جديد للبحث العلمي يركز على مجالات الذكاء الاصطناعي وتحليل البيانات والأمن السيبراني. ويهدف المركز إلى دعم مشاريع الطلبة والباحثين وتعزيز التعاون مع القطاع الصناعي، وتوفير بيئة بحثية متقدمة تساعد على تطوير حلول تقنية تخدم المجتمع.""",

    # 12 Fake
    """انتشر خبر يدعي أن علاجًا عشبيًا بسيطًا يمكنه القضاء على جميع أنواع السرطان خلال أيام قليلة، دون الحاجة إلى العلاج الكيميائي أو الجراحة. وحذرت مؤسسات طبية من هذه الادعاءات، مؤكدة أن علاج السرطان يحتاج إلى تشخيص طبي دقيق وخطة علاجية يحددها الأطباء المختصون.""",

    # 13 Real
    """أكدت وزارة الطاقة في تقرير رسمي بدء تنفيذ مشروع للطاقة الشمسية بقدرة إنتاجية عالية، بهدف زيادة الاعتماد على مصادر الطاقة المتجددة وتقليل كلفة استيراد الوقود، وذلك ضمن استراتيجية وطنية للطاقة تهدف إلى دعم الشبكة الكهربائية خلال السنوات القادمة.""",

    # 14 Fake
    """تداولت صفحات على الإنترنت معلومة تزعم أن الاستماع إلى موسيقى صاخبة لمدة عشر دقائق يزيد قوة العضلات بشكل مباشر ويجعل الشخص قادرًا على رفع أوزان مضاعفة. وأوضح خبراء الرياضة أن الموسيقى قد ترفع الحماس أثناء التمرين، لكنها لا تزيد القوة العضلية بشكل فوري أو سحري.""",

    # 15 Real
    """أعلنت دائرة الإحصاءات العامة في تقرير رسمي ارتفاع نسبة استخدام الخدمات الرقمية بين المواطنين خلال العام الحالي، خاصة في مجالات الدفع الإلكتروني والخدمات الحكومية والتعليم عن بعد. وأكد التقرير أن التحول الرقمي أصبح عنصرًا أساسيًا في الحياة اليومية مع الحاجة إلى تعزيز الوعي بالأمن الإلكتروني.""",

    # 16 Fake
    """انتشرت رسالة عبر تطبيقات المحادثة تزعم أن شرب كميات كبيرة جدًا من الماء دفعة واحدة ينظف الجسم من جميع السموم ويمنح طاقة خارقة. وأكد أطباء أن شرب الماء مهم للصحة، لكن الإفراط الشديد خلال وقت قصير قد يسبب اضطرابًا خطيرًا في أملاح الجسم.""",

    # 17 Real
    """أعلنت وزارة الزراعة في بيان رسمي عن خطة لدعم المزارعين تشمل توفير بذور محسنة وإرشادات فنية وبرامج تدريبية، بهدف تحسين الإنتاج الزراعي خلال الموسم الحالي وتقليل الخسائر الناتجة عن الظروف الجوية، إضافة إلى تعزيز الأمن الغذائي وتشجيع استخدام أساليب زراعية حديثة.""",

    # 18 Fake
    """زعمت منشورات متداولة أن استخدام الإنترنت يوميًا يؤدي حتمًا إلى فقدان البصر الكامل خلال أشهر قليلة. وأوضح أطباء العيون أن الاستخدام الطويل للشاشات قد يسبب إجهادًا وجفافًا مؤقتًا في العين، لكنه لا يؤدي إلى العمى الكامل كما تدعي تلك المنشورات.""",

    # 19 Real
    """أعلنت مؤسسة تعليمية عن إطلاق منصة مجانية للتعلم الإلكتروني توفر دورات في البرمجة واللغات والرياضيات والمهارات المهنية. وتهدف المنصة إلى دعم الطلبة والباحثين عن عمل من خلال محتوى تعليمي منظم، مع اختبارات قصيرة وشهادات إتمام لبعض المساقات.""",

    # 20 Fake
    """انتشرت شائعة تقول إن تناول ملعقة من العسل مع القرفة قبل النوم يعالج ارتفاع ضغط الدم نهائيًا خلال ثلاثة أيام. وأكد أطباء أن بعض الأغذية قد تكون مفيدة ضمن نظام صحي عام، لكنها لا تعوض الأدوية ولا تعالج الأمراض المزمنة بشكل نهائي دون متابعة طبية."""
]

# 1 = Real, 0 = Fake
expected_labels = [
    1, 0, 1, 0, 1,
    0, 1, 0, 1, 0,
    1, 0, 1, 0, 1,
    0, 1, 0, 1, 0
]


THRESHOLD_REAL = 0.40

correct = 0
results = []

for i, news in enumerate(news_list):
    result = predict(
        title=news,
        body="",
        threshold_real=THRESHOLD_REAL
    )

    expected = "Real" if expected_labels[i] == 1 else "Fake"
    predicted = result.get("prediction", "ERROR")

    is_correct = predicted == expected

    if is_correct:
        correct += 1

    results.append({
        "No": i + 1,
        "Expected": expected,
        "Predicted": predicted,
        "Correct": is_correct,
        "Confidence": result.get("confidence"),
        "Prob Fake": result.get("prob_fake"),
        "Prob Real": result.get("prob_real"),
        "Threshold": result.get("threshold_real"),
        "Words": result.get("word_count")
    })

    mark = "✓" if is_correct else "✗"

    print("=" * 70)
    print(f"NEWS {i + 1}")
    print(f"{mark} Expected: {expected} | Predicted: {predicted}")
    print(f"Confidence: {result.get('confidence')}")
    print(f"Prob Fake: {result.get('prob_fake')}")
    print(f"Prob Real: {result.get('prob_real')}")
    print(f"Threshold: {result.get('threshold_real')}")
    print(f"Words: {result.get('word_count')}")

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print(f"Correct predictions: {correct}/{len(news_list)}")
print(f"Accuracy on new test samples: {correct / len(news_list):.2%}")

results_df = pd.DataFrame(results)

try:
    display(results_df)
except NameError:
    print(results_df)

NEWS 1
✓ Expected: Real | Predicted: Real
Confidence: 0.4579
Prob Fake: 0.5421
Prob Real: 0.4579
Threshold: 0.4
Words: 51
NEWS 2
✓ Expected: Fake | Predicted: Fake
Confidence: 0.6938
Prob Fake: 0.6938
Prob Real: 0.3062
Threshold: 0.4
Words: 52
NEWS 3
✗ Expected: Real | Predicted: Fake
Confidence: 0.841
Prob Fake: 0.841
Prob Real: 0.159
Threshold: 0.4
Words: 47
NEWS 4
✓ Expected: Fake | Predicted: Fake
Confidence: 0.8999
Prob Fake: 0.8999
Prob Real: 0.1001
Threshold: 0.4
Words: 46
NEWS 5
✗ Expected: Real | Predicted: Fake
Confidence: 0.8936
Prob Fake: 0.8936
Prob Real: 0.1064
Threshold: 0.4
Words: 38
NEWS 6
✓ Expected: Fake | Predicted: Fake
Confidence: 0.7653
Prob Fake: 0.7653
Prob Real: 0.2347
Threshold: 0.4
Words: 46
NEWS 7
✓ Expected: Real | Predicted: Real
Confidence: 0.5224
Prob Fake: 0.4776
Prob Real: 0.5224
Threshold: 0.4
Words: 45
NEWS 8
✓ Expected: Fake | Predicted: Fake
Confidence: 0.7372
Prob Fake: 0.7372
Prob Real: 0.2628
Threshold: 0.4
Words: 46
NEWS 9
✗ Expected: Real | P

,No,Expected,Predicted,Correct,Confidence,Prob Fake,Prob Real,Threshold,Words
0,1,Real,Real,True,0.4579,0.5421,0.4579,0.4,51
1,2,Fake,Fake,True,0.6938,0.6938,0.3062,0.4,52
2,3,Real,Fake,False,0.8410,0.8410,0.1590,0.4,47
3,4,Fake,Fake,True,0.8999,0.8999,0.1001,0.4,46
4,5,Real,Fake,False,0.8936,0.8936,0.1064,0.4,38
5,6,Fake,Fake,True,0.7653,0.7653,0.2347,0.4,46
6,7,Real,Real,True,0.5224,0.4776,0.5224,0.4,45
7,8,Fake,Fake,True,0.7372,0.7372,0.2628,0.4,46
8,9,Real,Fake,False,0.8755,0.8755,0.1245,0.4,39
9,10,Fake,Fake,True,0.8120,0.8120,0.1880,0.4,44


In [ ]:
from google.colab import files
import shutil

shutil.make_archive("arabic_fakenews_stable", "zip", "arabic_fakenews_stable")

files.download("arabic_fakenews_stable.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
tokenizer.save_pretrained("arabic_fakenews_stable/")

('arabic_fakenews_stable/tokenizer_config.json',
 'arabic_fakenews_stable/tokenizer.json')